In [1]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

In [2]:
from datasets import config as ds_config
ds_config.AUDIO_DECODER_NAME = "soundfile"

In [3]:
data = load_dataset("openslr/librispeech_asr","clean",split="train.100")

data

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 28539
})

In [4]:
class CTCDataset(Dataset):
    def __init__(self, dataset, sample_rate=16000, n_fft=400, hop_length=160, n_mels=80):
        self.dataset = dataset
        self.transform = MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        audio = torch.tensor(item["audio"]["array"], dtype=torch.float32)
        mel = self.transform(audio)
        log_mel = AmplitudeToDB()(mel)
        label = item["text"]
        return log_mel, label

In [5]:
dataset = data.train_test_split(test_size=0.2)
train_dataset = CTCDataset(dataset["train"])
test_val = dataset["test"].train_test_split(test_size=0.5)
val_dataset = CTCDataset(test_val["train"])
test_dataset = CTCDataset(test_val["test"])

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel, stride=(1,1), padding=(0,0) ):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel, stride, padding)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.batch_norm(x)
        x = self.activation(x)
        return x

class CTCModel(nn.Module):
    def __init__(self,
                 n_mels=80,
                 rnn_hidden=512,
                 rnn_layers=5,
                 vocab_size=29,
                 dropout=0.1):
        super().__init__()
        
        self.conv = nn.Sequential(
            ConvBlock(1, 32, kernel=(41, 11), stride=(2, 2), padding=(20, 5)),
            ConvBlock(32, 32, kernel=(21, 11), stride=(2, 1), padding=(10, 5)),
        )

        rnn_input_size = (n_mels // 4) * 32
        self.rnn = nn.GRU(
            input_size=rnn_input_size,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if rnn_layers > 1 else 0
        )

        self.rnn_dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x, input_lengths):
        x = x.unsqueeze(1)

        x = self.conv(x)

        B,C,F,T = x.shape

        x = x.permute(0,3,1,2).reshape(B,T,C*F)

        output_lengths = (input_lengths + 1) //2
        x = nn.utils.rnn.pack_padded_sequence(
            x, output_lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        x, _ = self.rnn(x)
        x, _ = nn.utils.rnn.pad_packed_sequence(x, batch_first=True)

        x = self.rnn_dropout(x)

        x = self.fc(x)
        log_probs = x.permute(1,0,2).log_softmax(dim=2)

        return log_probs, output_lengths
    

In [13]:
class CharTokenizer:
    def __init__(self):
        self.blank = 0
        chars = list("abcdefghijklmnopqrstuvwxyz' ")
        self.char_to_idx = {char: idx + 1 for idx, char in enumerate(chars)}
        self.idx_to_char = {idx: char for char, idx in self.char_to_idx.items()}
        self.idx_to_char[self.blank] = ""
        self.vocab_size = len(chars) + 1


    def encode(self, text):
        return [self.char_to_idx[c] for c in text.lower() if c in self.char_to_idx]

    def decode_greedy(self, log_probs):
        indices = log_probs.argmax(dim=1)
        collapsed = []
        prev = -1

        for idx in indices:
            idx = idx.item()
            if idx != prev and idx != self.blank:
                collapsed.append(self.idx_to_char[idx])
            prev = idx

        return ''.join(collapsed)

In [14]:
def collate_fn(batch):
    mels, labels = zip(*batch)

    mel_lengths = torch.tensor([m.size(1) for m in mels])
    mels_padded = nn.utils.rnn.pad_sequence(
        [m.transpose(0,1) for m in mels],
        batch_first=True
    ).transpose(1,2)
    return mels_padded, mel_lengths, list(labels)

In [15]:
config = {
    "model": "CTCModel",
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-3,
    "batch_size": 32,
    "n_mels": 80,
    "rnn_hidden": 512,
    "rnn_layers": 5,
    "dropout": 0.1,
    "epochs": 20
}

tokenizer = CharTokenizer()
model = CTCModel(vocab_size=tokenizer.vocab_size).to("cuda")
ctc_loss = nn.CTCLoss(blank=tokenizer.blank, zero_infinity=True)

scaler = torch.amp.GradScaler()

train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn   
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)   
steps_per_epoch = len(train_loader)
warmup_steps = steps_per_epoch
total_steps = steps_per_epoch * config["epochs"]

optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])

wandb.init(project="learning-audio-ml", name="ctc_loss_cnn_gru", config=config)
wandb.watch(model, log="all")

In [16]:
from tqdm import tqdm
for epoch in range(config["epochs"]):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
        log_mel, input_lengths, labels = batch
        log_mel = log_mel.to("cuda")
        input_lengths = input_lengths.to("cuda")

        target_lengths = torch.tensor([len(tokenizer.encode(label)) for label in labels], dtype=torch.long).to("cuda")
        targets = torch.cat([torch.tensor(tokenizer.encode(label), dtype=torch.long) for label in labels]).to("cuda")

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            log_probs, output_lengths = model(log_mel, input_lengths)
            loss = ctc_loss(log_probs, targets, output_lengths, target_lengths)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{config['epochs']}, Loss: {avg_loss:.4f}")

    with torch.no_grad():
        pred = tokenizer.decode_greedy(log_probs[:, 0, :])
        print(f"  pred: '{pred}'")
        print(f"  true: '{labels[0]}'")

    wandb.log({"train_loss": avg_loss, "epoch": epoch})

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            log_mel, input_lengths, labels = batch
            log_mel = log_mel.to("cuda")
            input_lengths = input_lengths.to("cuda")

            target_lengths = torch.tensor([len(tokenizer.encode(label)) for label in labels], dtype=torch.long).to("cuda")
            targets = torch.cat([torch.tensor(tokenizer.encode(label), dtype=torch.long) for label in labels]).to("cuda")

            with torch.amp.autocast("cuda"):
                log_probs, output_lengths = model(log_mel, input_lengths)
                loss = ctc_loss(log_probs, targets, output_lengths, target_lengths)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Validation Loss: {avg_val_loss:.4f}")
    wandb.log({"val_loss": avg_val_loss, "epoch": epoch})

Epoch 1 Train:   0%|          | 0/714 [00:00<?, ?it/s]/home/ubuntu/miniforge3/envs/audio-ml/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Epoch 1 Train: 100%|█████████▉| 713/714 [06:45<00:00,  1.87it/s]/home/ubuntu/miniforge3/envs/audio-ml/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable

Epoch 1/20, Loss: 2.5740
  pred: 'an in wi thero lot  ri yurlaton s oti ousut mras pe t erlingrad ther brtit mas be tl lors ol eforin tisol so the epol o ot betom ti te an to  ther rs l isol te ood '
  true: 'IN ENGLAND THERE ARE LOCAL REGULATIONS ON THE USE OF MASHED POTATO IN BREAD THEIR BREAD MUST BE TWELVE HOURS OLD BEFORE IT IS SOLD SO THAT PEOPLE WILL NOT BE TEMPTED TO EAT TOO MUCH THE RESULT IS SELDOM PALATABLE'


Validation: 100%|██████████| 90/90 [00:31<00:00,  2.86it/s]


Validation Loss: 1.5785


Epoch 2 Train: 100%|██████████| 714/714 [05:51<00:00,  2.03it/s]


...

Validation Loss: 0.3744


Epoch 20 Train: 100%|██████████| 714/714 [05:50<00:00,  2.04it/s]


Epoch 20/20, Loss: 0.2109
  pred: 'so be it let us raise the baricade to a height of twenty feet and let us all remain in it citicens letus offer the protests of corpsess'
  true: 'SO BE IT LET US RAISE THE BARRICADE TO A HEIGHT OF TWENTY FEET AND LET US ALL REMAIN IN IT CITIZENS LET US OFFER THE PROTESTS OF CORPSES'


Validation: 100%|██████████| 90/90 [00:28<00:00,  3.14it/s]

Validation Loss: 0.3740


In [17]:
from tqdm import tqdm

def compute_wer(pred, target):
    pred_words = pred.split()
    target_words = target.split()
    d = [[0] * (len(target_words) + 1) for _ in range(len(pred_words) + 1)]
    for i in range(len(pred_words) + 1):
        d[i][0] = i
    for j in range(len(target_words) + 1):
        d[0][j] = j
    for i in range(1, len(pred_words) + 1):
        for j in range(1, len(target_words) + 1):
            if pred_words[i-1] == target_words[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = 1 + min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[-1][-1], len(target_words)

def compute_cer(pred, target):
    pred_chars = list(pred)
    target_chars = list(target)
    d = [[0] * (len(target_chars) + 1) for _ in range(len(pred_chars) + 1)]
    for i in range(len(pred_chars) + 1):
        d[i][0] = i
    for j in range(len(target_chars) + 1):
        d[0][j] = j
    for i in range(1, len(pred_chars) + 1):
        for j in range(1, len(target_chars) + 1):
            if pred_chars[i-1] == target_chars[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = 1 + min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[-1][-1], len(target_chars)

test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

model.eval()
total_wer_errors = 0
total_wer_words = 0
total_cer_errors = 0
total_cer_chars = 0
samples = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Evaluation"):
        log_mel, input_lengths, labels = batch
        log_mel = log_mel.to("cuda")
        input_lengths = input_lengths.to("cuda")

        with torch.amp.autocast("cuda"):
            log_probs, output_lengths = model(log_mel, input_lengths)

        for i in range(log_probs.size(1)):
            pred = tokenizer.decode_greedy(log_probs[:, i, :])
            target = labels[i].lower()

            wer_errors, wer_words = compute_wer(pred, target)
            cer_errors, cer_chars = compute_cer(pred, target)

            total_wer_errors += wer_errors
            total_wer_words += wer_words
            total_cer_errors += cer_errors
            total_cer_chars += cer_chars

            if len(samples) < 10:
                samples.append((pred, target))

wer = total_wer_errors / total_wer_words * 100
cer = total_cer_errors / total_cer_chars * 100

print(f"Test WER: {wer:.2f}%")
print(f"Test CER: {cer:.2f}%")
print(f"\n{'='*80}")
print(f"{'PREDICTION':<40} | {'GROUND TRUTH':<40}")
print(f"{'='*80}")
for pred, target in samples:
    print(f"{pred[:40]:<40} | {target[:40]:<40}")


Test Evaluation: 100%|██████████| 90/90 [01:23<00:00,  1.07it/s]

Test WER: 34.52%
Test CER: 10.81%

PREDICTION                               | GROUND TRUTH                            
pernise was not a man to have such a com | bernajoux was not a man to have such a c
and he not even to dream of my secredeed | and he not even to dream of my secret de
as the sunwint downe ehend the low hills | as the sun went down behind the low hill
but a fewmin uts before would haveriskan | but a few minutes before would have risk
others in the sam roe seats of his own w | others in the same row of seats as his o
what i had read of his writing disposedo | what i had read of his writing disposed 
were missing hin goon back into the land | or missing him gloomed back into the lan
much to the amusement of some children l | much to the amusement of some children r
and she waped the gobles she was holding | and she waved the goblet she was holding
after all ar colected the paper as secue | after all are collected the paper is sec
